# 卒研の実験テンプレート(結果をGoogle Driveに保存する)

このnotebookは、Colabで実験を行い、結果を **Google Drive** に保存し、卒研repoにアップロードするファイルを書き出すまでのテンプレートです。
手順の説明は `technical-handbook/colab/experiment-workflow.md` を読んでください。

**使い方**

1. このnotebookを開き、「ファイル」→「GitHubにコピーを保存」で**自分の卒研repo**の `notebooks/` に保存する(`lab-core` は直接編集しない)
2. セル1の `STUDENT_ID` を自分の学籍番号に変える
3. 上から順に実行する
4. 自分の研究では、セル2の `config` とセル3の `run_q_learning()` を自分の環境・手法に置き換え、記録する指標を変える

例として、`FrozenLake-v1` を表形式Q学習で3つのseedについて解きます。

## 0. ライブラリの準備

Colabに入っていない場合に備えてインストールする(入っていれば何もしない)。

In [ ]:
!pip install -q gymnasium

## 1. 保存先を決める

Google Driveをマウントし、`マイドライブ/cpai-results/<学籍番号>/<日付_時刻_実験名>/` を作る。許可画面が出たら、Colabで使っているGoogleアカウントを選んで許可する。

In [ ]:
import os, json, datetime, platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/cpai-results'
except ImportError:
    BASE_DIR = './cpai-results'   # Colab以外(ローカル)で実行したとき

STUDENT_ID = 'e16xxxxx'          # 自分の学籍番号(卒研repo名)に変える
EXPERIMENT = 'qlearning_frozenlake'  # 手法_環境 を半角英小文字と _ で

RUN_NAME = f"{datetime.datetime.now():%Y-%m-%d_%H%M}_{EXPERIMENT}"
RUN_DIR = os.path.join(BASE_DIR, STUDENT_ID, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=False)   # 同名フォルダがあればエラー(上書き防止)
print('保存先:', RUN_DIR)

## 2. 実験条件を記録する

**結果より先に条件を保存する。** 条件を変えるときは、コード中の数値ではなくこの `config` を変える。

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib

config = {
    'env_id': 'FrozenLake-v1',
    'env_kwargs': {'is_slippery': True},
    'algorithm': 'q_learning',
    'n_episodes': 2000,
    'alpha': 0.1,
    'gamma': 0.99,
    'epsilon': 0.1,
    'seeds': [0, 1, 2],
    'versions': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'gymnasium': gym.__version__,
        'matplotlib': matplotlib.__version__,
    },
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
}
with open(os.path.join(RUN_DIR, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

## 3. 実験を実行し、ログを保存する

エピソードごとに1行をCSVに記録する。seedごとにファイルへ書き込むので、途中で切断されても終わったseedの結果は残る。

In [ ]:
import csv

def run_q_learning(cfg, seed):
    env = gym.make(cfg['env_id'], **cfg['env_kwargs'])
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    rows = []
    for ep in range(cfg['n_episodes']):
        s, _ = env.reset(seed=seed if ep == 0 else None)
        total, steps, done = 0.0, 0, False
        while not done:
            if rng.random() < cfg['epsilon']:
                a = int(rng.integers(env.action_space.n))
            else:
                a = int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))
            s2, r, terminated, truncated, _ = env.step(a)
            target = r + (0.0 if terminated else cfg['gamma'] * Q[s2].max())
            Q[s, a] += cfg['alpha'] * (target - Q[s, a])
            s, total, steps = s2, total + r, steps + 1
            done = terminated or truncated
        rows.append({'seed': seed, 'episode': ep, 'return': total, 'steps': steps})
    env.close()
    return rows, Q

metrics_path = os.path.join(RUN_DIR, 'metrics.csv')
with open(metrics_path, 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=['seed', 'episode', 'return', 'steps']).writeheader()

for seed in config['seeds']:
    rows, Q = run_q_learning(config, seed)
    with open(metrics_path, 'a', newline='', encoding='utf-8') as f:
        csv.DictWriter(f, fieldnames=['seed', 'episode', 'return', 'steps']).writerows(rows)
    np.save(os.path.join(RUN_DIR, f'q_table_seed{seed}.npy'), Q)
    print(f'seed {seed} 完了')

## 4. 図と要約を作る

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(metrics_path)
curve = df.groupby('episode')['return'].mean().rolling(100, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curve)
ax.set_xlabel('episode')
ax.set_ylabel('success rate (moving average, 3 seeds)')
ax.set_title(RUN_NAME)
fig.tight_layout()
fig.savefig(os.path.join(RUN_DIR, 'learning_curve.png'), dpi=150)
plt.show()

last = df[df['episode'] >= config['n_episodes'] - 100]
summary = last.groupby('seed')['return'].mean().rename('success_rate_last100').reset_index()
summary.to_csv(os.path.join(RUN_DIR, 'summary.csv'), index=False)
print(summary)

## 5. 卒研repo用のファイルを書き出す

`for_github/` の中の `configs`・`results`・`figures` の3フォルダを、卒研repoの「Add file」→「Upload files」に**まとめて**ドラッグ&ドロップする。

In [ ]:
import shutil

GH_DIR = os.path.join(RUN_DIR, 'for_github')
for sub in ['configs', 'results', 'figures']:
    os.makedirs(os.path.join(GH_DIR, sub), exist_ok=True)
shutil.copy(os.path.join(RUN_DIR, 'config.json'),        os.path.join(GH_DIR, 'configs', f'{RUN_NAME}.json'))
shutil.copy(os.path.join(RUN_DIR, 'summary.csv'),        os.path.join(GH_DIR, 'results', f'{RUN_NAME}_summary.csv'))
shutil.copy(os.path.join(RUN_DIR, 'learning_curve.png'), os.path.join(GH_DIR, 'figures', f'{RUN_NAME}_learning_curve.png'))
for root, _, files in os.walk(GH_DIR):
    for name in files:
        print(os.path.relpath(os.path.join(root, name), GH_DIR))

## 6. 最後にやること

- [ ] 「ファイル」→「GitHubにコピーを保存」で、このnotebookを自分の卒研repoに保存した
- [ ] `for_github/` の3フォルダを卒研repoにアップロードした
- [ ] 週報(`docs/weekly/`)に、実験内容・結果・保存先を書いた